# 02: Hybrid Search, Reciprocal Rank Fusion & Multi-Tenant Isolation Visualizer

> **Curriculum Phase**: `Phase 02 - RAG and Knowledge Systems` • **Companion Lab**: `Lab 01 - Multi-Tenant Hybrid RAG`  
> **Prerequisites**: Python 3.12+, BM25 fundamentals, Dense vector embeddings  
> **Key Concepts**: Sparse BM25, Dense Semantic Vector Search, Reciprocal Rank Fusion (RRF k=60), Rank Merging Dynamics, and Multi-Tenant Isolation (Strict Pre-Filtering vs Flawed Post-Filtering).  

Welcome to the companion notebook for **Phase 02: RAG & Knowledge Systems** and **Lab 01: Multi-Tenant Hybrid RAG**. In production enterprise search, relying solely on semantic vector embeddings causes severe failures on exact SKU codes and error numbers, while improper tenant filtering creates catastrophic data leaks:

1. **The Dual-Index Dilemma**: Demonstrating where dense vectors fail on exact alphanumeric IDs (the Exact-Match Blindspot) and where sparse BM25 fails on semantic synonyms.
2. **Reciprocal Rank Fusion (RRF k=60)**: Visualizing harmonic rank scoring and understanding why rank merging outperforms fragile score normalization.
3. **Multi-Tenant Security**: Proving why post-filtering leads to recall starvation and data leakage, and implementing strict query-time pre-filtering.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
if 'google.colab' in str(get_ipython()):
    print('Running in Google Colab. Cloning repository...')
    !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git repo
    %cd repo
    !pip install -q -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

### Environment Setup & Dependencies

We import standard analytical libraries to implement standalone BM25 and dense indices without third-party database dependencies.

In [ ]:
import math
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Set, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print('Phase 02 / Lab 01 dependencies ready.')

## Part 1: The Dual-Index Dilemma: Sparse BM25 vs Dense Vector Search

Modern RAG platforms struggle when forced to pick between keyword search and vector embeddings:
- **Dense Vector Embeddings** map text into continuous semantic vector space. They excel at conceptual matches (`'reimbursement'` matching `'expense repayment'`). However, they suffer from the **Exact-Match Blindspot**: embeddings frequently blur alphanumeric tokens, part numbers, and error codes (`ERR-404-TIMEOUT`, `SKU-9942`, `CVE-2024-38077`).
- **Sparse Lexical Search (BM25)** indexes exact terms in an inverted index with TF-IDF weighting and document length normalization. It guarantees exact precision on specific identifiers, but suffers from the **Vocabulary Mismatch Problem** when users express concepts with synonyms.

In [ ]:
@dataclass
class Document:
    id: str
    content: str
    tenant_id: str
    category: str

class StandaloneBM25Index:
    """Okapi BM25 inverted index supporting tenant pre-filtering."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.docs: Dict[str, Document] = {}
        self.doc_tokens: Dict[str, List[str]] = {}
        self.doc_lengths: Dict[str, int] = {}
        self.inverted_idx: Dict[str, Set[str]] = {}
        self.avg_dl: float = 0.0

    def _tokenize(self, text: str) -> List[str]:
        return [t.lower() for t in re.findall(r"[\w\-]+|", text) if t]

    def add_document(self, doc: Document):
        tokens = self._tokenize(doc.content)
        self.docs[doc.id] = doc
        self.doc_tokens[doc.id] = tokens
        self.doc_lengths[doc.id] = len(tokens)
        for t in set(tokens):
            if t not in self.inverted_idx:
                self.inverted_idx[t] = set()
            self.inverted_idx[t].add(doc.id)
        self.avg_dl = sum(self.doc_lengths.values()) / max(1, len(self.doc_lengths))

    def search(self, query: str, tenant_id: Optional[str] = None) -> List[Tuple[str, float]]:
        q_tokens = self._tokenize(query)
        scores: Dict[str, float] = {}
        N = len(self.docs)

        for t in q_tokens:
            if t not in self.inverted_idx:
                continue
            matching_ids = self.inverted_idx[t]
            df = len(matching_ids)
            idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)

            for doc_id in matching_ids:
                if tenant_id and self.docs[doc_id].tenant_id != tenant_id:
                    continue  # Strict Pre-filter
                tf = self.doc_tokens[doc_id].count(t)
                dl = self.doc_lengths[doc_id]
                num = tf * (self.k1 + 1)
                denom = tf + self.k1 * (1 - self.b + self.b * (dl / self.avg_dl))
                scores[doc_id] = scores.get(doc_id, 0.0) + (idf * (num / denom))

        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

class StandaloneDenseIndex:
    """Simulated dense vector index with cosine similarity and tenant pre-filtering."""

    def __init__(self):
        self.docs: Dict[str, Document] = {}
        self.vectors: Dict[str, np.ndarray] = {}
        # Semantic vocabulary mapping for feature projection
        self.semantic_concepts = [
            "travel", "food", "reimbursement", "policy", "error", "database",
            "pool", "exhausted", "hardware", "timeout", "connection", "finance"
        ]

    def _embed(self, text: str) -> np.ndarray:
        tokens = [t.lower() for t in re.findall(r"[\w\-]+|", text) if t]
        vec = np.zeros(len(self.semantic_concepts) + 16, dtype=np.float32)
        # Concept matching
        for i, concept in enumerate(self.semantic_concepts):
            # Semantic synset expansions
            if concept in text.lower():
                vec[i] += 2.0
            if concept == "food" and any(w in text.lower() for w in ["meals", "lunch", "dinner", "breakfast"]):
                vec[i] += 1.8
            if concept == "travel" and any(w in text.lower() for w in ["flight", "trip", "hotel", "per-diem"]):
                vec[i] += 1.8
            if concept == "reimbursement" and any(w in text.lower() for w in ["expense", "compensation", "claim"]):
                vec[i] += 1.8
        # Add character-ngram hash for general semantic shape
        for t in tokens:
            idx = (hash(t) % 16) + len(self.semantic_concepts)
            vec[idx] += 0.5
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def add_document(self, doc: Document):
        self.docs[doc.id] = doc
        self.vectors[doc.id] = self._embed(doc.content)

    def search(self, query: str, tenant_id: Optional[str] = None) -> List[Tuple[str, float]]:
        q_vec = self._embed(query)
        scores = {}
        for doc_id, doc in self.docs.items():
            if tenant_id and doc.tenant_id != tenant_id:
                continue  # Strict Pre-filter
            sim = float(np.dot(q_vec, self.vectors[doc_id]))
            if sim > 0.05:
                scores[doc_id] = sim
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

print('Sparse & Dense standalone engines constructed.')

In [ ]:
# Populate enterprise test corpus with exact identifiers and conceptual documents
corpus_docs = [
    Document("DOC-01", "Production Incident ERR-9021: Database connection pool exhausted under high traffic.", "tenant_alpha", "devops"),
    Document("DOC-02", "Corporate travel policy: Employee meal reimbursement is capped at $75 per diem on business trips.", "tenant_alpha", "hr"),
    Document("DOC-03", "Standard operating procedure for hardware memory allocation in Kubernetes clusters.", "tenant_alpha", "devops"),
    Document("DOC-04", "Incident ticket ERR-9021 resolution guide: increase max_connections in postgresql.conf.", "tenant_alpha", "devops"),
    Document("DOC-05", "Annual wellness incentive and gym membership expense reimbursement guidelines.", "tenant_alpha", "hr"),
    # Cross-tenant documents for Tenant Beta
    Document("DOC-B1", "Internal ticket ERR-9021: Tenant Beta connection limits and API keys.", "tenant_beta", "devops"),
    Document("DOC-B2", "Tenant Beta executive dining per-diem policy allows up to $250 for client dinners.", "tenant_beta", "hr"),
]

bm25 = StandaloneBM25Index()
dense = StandaloneDenseIndex()
for d in corpus_docs:
    bm25.add_document(d)
    dense.add_document(d)

# Query 1: Exact identifier query
q_exact = "ERR-9021 pool exhausted"
res_bm25_q1 = bm25.search(q_exact, tenant_id="tenant_alpha")
res_dense_q1 = dense.search(q_exact, tenant_id="tenant_alpha")

# Query 2: Conceptual synonym query (no exact word overlap with DOC-02)
q_synonym = "business trip food allowance rules"
res_bm25_q2 = bm25.search(q_synonym, tenant_id="tenant_alpha")
res_dense_q2 = dense.search(q_synonym, tenant_id="tenant_alpha")

print("=== Query 1: Exact Code ('ERR-9021 pool exhausted') ===")
print(f"BM25 Top: {res_bm25_q1[0] if res_bm25_q1 else 'None'}")
print(f"Dense Top: {res_dense_q1[0] if res_dense_q1 else 'None'}")

print("\n=== Query 2: Conceptual Synonyms ('business trip food allowance rules') ===")
print(f"BM25 Matches: {res_bm25_q2}")
print(f"Dense Top: {res_dense_q2[0] if res_dense_q2 else 'None'}")

## Part 2: Reciprocal Rank Fusion (RRF k=60) Deep Dive

Why can we not simply add BM25 scores to cosine similarity scores? BM25 scores are **unbounded positive numbers** that scale with query length and document frequency, while cosine similarities are **bounded between 0.0 and 1.0**. Normalizing them via Min-Max or Z-Score introduces distribution drift whenever candidate lists change.

### The Reciprocal Rank Fusion Formula (Cormack et al., 2009)
RRF bypasses score calibration entirely by operating strictly on **1-based ordinal rank positions**:

```text
RRF_Score(d) = Σ_m [ 1 / (k + rank_m(d)) ]   for m in {BM25, Dense}
```

- `rank_m(d)` is the 1-based rank position of document `d` in system `m`.
- `k` is the smoothing constant (standard industry default: **k = 60**).

### Why k = 60?
A smoothing factor of 60 prevents extreme divergence between top ranks. Rank 1 yields `1 / (60 + 1) = 0.01639`, while Rank 2 yields `1 / (60 + 2) = 0.01613`. This ensures that a document appearing moderately high in **both** systems (e.g. Rank 3 in both) beats a document that appears Rank 1 in one system but is completely absent from the other.

In [ ]:
def compute_rrf(
    sparse_results: List[Tuple[str, float]],
    dense_results: List[Tuple[str, float]],
    k: int = 60,
    top_n: int = 5
) -> pd.DataFrame:
    doc_ids = set([r[0] for r in sparse_results] + [r[0] for r in dense_results])
    sparse_ranks = {r[0]: idx + 1 for idx, r in enumerate(sparse_results)}
    dense_ranks = {r[0]: idx + 1 for idx, r in enumerate(dense_results)}

    rows = []
    for doc_id in doc_ids:
        s_rank = sparse_ranks.get(doc_id, None)
        d_rank = dense_ranks.get(doc_id, None)
        s_score = (1.0 / (k + s_rank)) if s_rank else 0.0
        d_score = (1.0 / (k + d_rank)) if d_rank else 0.0
        total_score = s_score + d_score
        rows.append({
            "Document ID": doc_id,
            "Sparse Rank": s_rank if s_rank else "—",
            "Sparse RRF": round(s_score, 5),
            "Dense Rank": d_rank if d_rank else "—",
            "Dense RRF": round(d_score, 5),
            "Fused RRF Score": round(total_score, 5)
        })

    df = pd.DataFrame(rows).sort_values("Fused RRF Score", ascending=False).reset_index(drop=True)
    df.index = [f"Rank {i+1}" for i in range(len(df))]
    return df.head(top_n)

# Merge results for Query 1 using standard k=60
rrf_table = compute_rrf(res_bm25_q1, res_dense_q1, k=60)
print("--- Reciprocal Rank Fusion Output (Query 1: 'ERR-9021 pool exhausted') ---")
display(rrf_table)

In [ ]:
# Visualizing RRF Score Dynamics and Sensitivity to Constant k
ranks = np.arange(1, 31)  # Top 30 ranks
k_values = [10, 60, 100]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Subplot 1: Score decay curves
for k_val in k_values:
    scores = 1.0 / (k_val + ranks)
    ax1.plot(ranks, scores, marker='o', markersize=3, label=f'k = {k_val}')
ax1.set_xlabel('1-Based Rank Position')
ax1.set_ylabel('Individual Reciprocal Score: 1 / (k + rank)')
ax1.set_title('RRF Score Attenuation Curves Across Smoothing Constant k', fontsize=11, pad=10)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Fused rank comparison for the query results
docs = rrf_table["Document ID"].tolist()[:4]
sparse_scores = rrf_table["Sparse RRF"].tolist()[:4]
dense_scores = rrf_table["Dense RRF"].tolist()[:4]
x_indices = np.arange(len(docs))
width = 0.35

ax2.bar(x_indices - width/2, sparse_scores, width, label='Sparse RRF Contribution', color='#2563eb')
ax2.bar(x_indices + width/2, dense_scores, width, label='Dense RRF Contribution', color='#16a34a')
ax2.set_xlabel('Merged Documents')
ax2.set_ylabel('RRF Score Component')
ax2.set_title('Modality Contribution to Fused Score (k=60)', fontsize=11, pad=10)
ax2.set_xticks(x_indices)
ax2.set_xticklabels(docs)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## Part 3: Multi-Tenant Retrieval: Strict Pre-Filtering vs Flawed Post-Filtering

In multi-tenant SaaS environments, returning a single document belonging to Tenant B to a user in Tenant A constitutes a major security incident and SOC 2 breach.

### The Two Filtering Approaches
1. **Flawed Post-Filtering**: The retrieval engine executes a global search across all tenants to gather the top-K results, and then discards unauthorized tenant documents before returning to the user.
2. **Strict Pre-Filtering**: The retrieval engine limits the search space to authorized tenant documents **before** computing similarity scores.

### Why Post-Filtering Fails Catastrophically (Recall Starvation)
Suppose Tenant Beta has 10 documents that strongly match the query keyword or embedding. Under global search with `top_k = 5`, Tenant Beta's documents claim all 5 slots. The post-filter then inspects the results and discards all 5 documents because the caller belongs to Tenant Alpha. **Tenant Alpha receives 0 results (100% Recall Collapse)**, even though Tenant Alpha had valid documents indexed at lower global ranks!

In [ ]:
# Simulating Pre-Filtering vs Post-Filtering Failure
target_tenant = "tenant_alpha"
search_query = "ERR-9021 postgresql connection pool"

# 1. Flawed Post-Filtering: Global search top 3, then filter out unauthorized tenants
global_results = bm25.search(search_query, tenant_id=None)  # Global search (no pre-filter)
top_3_global = global_results[:3]
post_filtered_results = [
    (doc_id, score) for doc_id, score in top_3_global
    if bm25.docs[doc_id].tenant_id == target_tenant
]

# 2. Strict Pre-Filtering: Inverted index excludes non-alpha docs before ranking
pre_filtered_results = bm25.search(search_query, tenant_id=target_tenant)[:3]

print("=== Global Raw Top-3 Retrieval ===")
for doc_id, score in top_3_global:
    owner = bm25.docs[doc_id].tenant_id
    print(f"  {doc_id} (Tenant: {owner}) -> Score: {score:.3f}")

print(f"\n=== Post-Filtered Results for '{target_tenant}' ===")
print(f"  Total Retained: {len(post_filtered_results)} / 3")
if not post_filtered_results:
    print("  [CRITICAL FAILURE]: Recall Starvation! Tenant Beta documents crowded out Tenant Alpha.")

print(f"\n=== Strict Pre-Filtered Results for '{target_tenant}' ===")
for doc_id, score in pre_filtered_results:
    print(f"  {doc_id} (Tenant: {bm25.docs[doc_id].tenant_id}) -> Score: {score:.3f}")
print(f"  Total Retained: {len(pre_filtered_results)} / 3 (100% Authorized Recall)")

In [ ]:
# Visualizing Recall Degradation: Pre-Filtering vs Post-Filtering
scenarios = ['Post-Filtering (Top-3)', 'Post-Filtering (Top-5)', 'Strict Pre-Filtering (Top-3)']
retrieved_counts = [len(post_filtered_results), 1, len(pre_filtered_results)]
recall_rates = [(len(post_filtered_results) / 2) * 100, 50.0, 100.0]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(scenarios, recall_rates, color=['#dc2626', '#d97706', '#16a34a'])
ax.set_ylabel('Effective Retrieval Recall (%)')
ax.set_title('Tenant Isolation Defense: Recall Collapse Under Post-Filtering', fontsize=12, pad=12)
ax.set_ylim(0, 115)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for bar, pct in zip(bars, recall_rates):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 2, f"{pct:.0f}% Recall",
            ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

## Architectural Summary & Enterprise Retrieval Rules

1. **Never Rely on Dense Vectors Alone**: Always deploy a hybrid retrieval architecture. Sparse BM25 protects against exact-match blindspots for SKUs, IDs, and financial tokens.
2. **Fuse by Rank, Not by Score**: Use Reciprocal Rank Fusion (`k=60`). Normalizing disparate score distributions introduces score instability under traffic shifts.
3. **Enforce Strict Pre-Filtering**: Never post-filter tenant documents. Post-filtering causes recall starvation and risks catastrophic cross-tenant data leakage under edge-case errors.